In [2]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import neighbors, datasets


## Load bộ dữ liệu Iris và xem vài mẫu đầu tiên

Iris là bộ dữ liệu rất nổi tiếng trong Machine Learning. Mỗi bông hoa được mô tả bởi 4 đặc trưng:
- chiều dài đài hoa,
- chiều rộng đài hoa,
- chiều dài cánh hoa,
- chiều rộng cánh hoa.

Cell dưới đây load dữ liệu và in ra vài mẫu thuộc từng class để mình có cảm giác trực quan hơn về dữ liệu đầu vào.

In [3]:
iris = datasets.load_iris()
iris_X = iris.data
iris_y = iris.target
print('Number of classes: %d' % len(np.unique(iris_y)))
print('Number of data points: %d' % len(iris_y))

X0 = iris_X[iris_y == 0, :]
print('\nSamples from class 0:\n', X0[:5, :])
X1 = iris_X[iris_y == 1, :]
print('\nSamples from class 1:\n', X1[:5, :])
X2 = iris_X[iris_y == 2, :]
print('\nSamples from class 2:\n', X2[:5, :])


Number of classes: 3
Number of data points: 150

Samples from class 0:
 [[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]
 [4.6 3.1 1.5 0.2]
 [5.  3.6 1.4 0.2]]

Samples from class 1:
 [[7.  3.2 4.7 1.4]
 [6.4 3.2 4.5 1.5]
 [6.9 3.1 4.9 1.5]
 [5.5 2.3 4.  1.3]
 [6.5 2.8 4.6 1.5]]

Samples from class 2:
 [[6.3 3.3 6.  2.5]
 [5.8 2.7 5.1 1.9]
 [7.1 3.  5.9 2.1]
 [6.3 2.9 5.6 1.8]
 [6.5 3.  5.8 2.2]]


## Tách training set và test set

Ở đây ta chia 150 điểm dữ liệu thành 2 phần:
- `100` điểm cho training,
- `50` điểm cho test.

Bài gốc chia ngẫu nhiên, nên kết quả có thể thay đổi đôi chút giữa các lần chạy. Mình thêm `random_state` để notebook cho kết quả ổn định hơn khi bạn chạy lại nhiều lần.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    iris_X, iris_y, test_size=50, random_state=42, stratify=iris_y
)

print('Training size: %d' % len(y_train))
print('Test size    : %d' % len(y_test))


Training size: 100
Test size    : 50


## Thử trường hợp đơn giản nhất: 1-NN

Đây là phiên bản dễ hiểu nhất của KNN:
- với mỗi điểm test,
- ta chỉ nhìn đúng **1 điểm training gần nhất**, 
- rồi lấy label của điểm đó làm dự đoán.

Trong code, `p = 2` nghĩa là dùng khoảng cách Euclid.

In [5]:
clf = neighbors.KNeighborsClassifier(n_neighbors=1, p=2)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Print results for 20 test data points:')
print('Predicted labels:', y_pred[20:40])
print('Ground truth    :', y_test[20:40])


Print results for 20 test data points:
Predicted labels: [2 1 0 2 0 2 2 0 0 2 2 2 0 1 0 0 2 1 1 1]
Ground truth    : [2 1 0 2 0 2 2 0 0 2 2 2 0 1 0 0 2 1 1 1]


## Đánh giá độ chính xác của 1-NN

Sau khi có dự đoán, ta cần xem mô hình làm tốt đến đâu.

Trong bài gốc, tác giả dùng `accuracy_score`, tức là lấy số điểm dự đoán đúng chia cho tổng số điểm trong tập test.

In [6]:
from sklearn.metrics import accuracy_score

print('Accuracy of 1NN: %.2f %%' % (100 * accuracy_score(y_test, y_pred)))


Accuracy of 1NN: 94.00 %


## Tăng số láng giềng lên 10 và dùng major voting

Nếu chỉ nhìn đúng 1 điểm gần nhất thì mô hình có thể bị nhiễu nếu điểm đó là outlier.

Một cách khắc phục là tăng lên `10` láng giềng gần nhất, rồi để các láng giềng này bỏ phiếu theo đa số. Cách đó gọi là **major voting**.

In [7]:
clf = neighbors.KNeighborsClassifier(n_neighbors=10, p=2)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Accuracy of 10NN with major voting: %.2f %%' % (100 * accuracy_score(y_test, y_pred)))


Accuracy of 10NN with major voting: 96.00 %


## Gán trọng số theo khoảng cách

Trong major voting ở trên, 10 láng giềng đều có lá phiếu như nhau. Nhưng trực giác cho thấy điều này chưa thật công bằng:
- điểm ở rất gần test point nên được tin nhiều hơn,
- điểm ở xa hơn nên có ảnh hưởng nhỏ hơn.

Scikit-learn hỗ trợ sẵn ý tưởng này bằng `weights='distance'`, nghĩa là điểm càng gần thì trọng số càng lớn.

In [8]:
clf = neighbors.KNeighborsClassifier(n_neighbors=10, p=2, weights='distance')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Accuracy of 10NN (1/distance weights): %.2f %%' % (100 * accuracy_score(y_test, y_pred)))


Accuracy of 10NN (1/distance weights): 96.00 %


## Tự định nghĩa hàm trọng số

Ngoài hai chế độ mặc định là `uniform` và `distance`, mình còn có thể tự định nghĩa cách đánh trọng số.

Bài gốc minh họa một hàm dạng Gaussian:

`w_i = exp(-||x - x_i||^2 / sigma^2)`

Ý nghĩa của công thức này vẫn giống nhau: điểm càng gần thì trọng số càng cao, chỉ khác là cách giảm trọng số theo khoảng cách mượt hơn.

In [9]:
def myweight(distances):
    sigma2 = 0.5  # we can change this number
    return np.exp(-(distances ** 2) / sigma2)

clf = neighbors.KNeighborsClassifier(n_neighbors=10, p=2, weights=myweight)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('Accuracy of 10NN (customized weights): %.2f %%' % (100 * accuracy_score(y_test, y_pred)))


Accuracy of 10NN (customized weights): 96.00 %


## Ghi chú cuối cùng

Khi chạy xong notebook này,nên để ý ba điều:
- `K` nhỏ thì mô hình nhạy với nhiễu hơn,
- `K` lớn hơn có thể ổn định hơn nhưng cũng có lúc làm mờ ranh giới giữa các class,
- cách đánh trọng số ảnh hưởng trực tiếp đến kết quả dự đoán.
